In [3]:
import pandas as pd
import statsmodels.api as sm

# =========================
# 1. Load Data
# =========================
temp_df = pd.read_csv("temperature_clean.csv")
urban_df = pd.read_csv("urban_surface.csv")

In [5]:
temp_df
urban_df

,neighbourhood_id,latitude,longitude,distance_from_center_km,tree_cover_pct,asphalt_pct,building_density,median_income,population_density,heat_retention_factor,infrastructure_quality_index,social_vulnerability_index
0,1,22.346937,73.255294,9.332579,32.950793,78.999617,0.356619,1.027265e+06,7087.025990,0.866993,1.216323,0.606696
1,2,22.296139,73.333953,17.000015,5.690045,90.000000,0.300000,4.484904e+05,2256.167960,0.841827,0.572084,0.351410
2,3,22.359015,73.069315,13.686416,21.070189,90.000000,0.300000,6.216405e+05,7959.534652,1.054572,0.571257,0.451236
3,4,22.429042,73.226238,14.418873,16.839119,90.000000,0.300000,6.681284e+05,3232.654634,1.082590,0.512108,0.774405
4,5,22.288468,73.129149,6.140464,38.371173,67.672207,0.574361,1.224885e+06,9348.087316,0.812634,1.456501,1.423620
...,...,...,...,...,...,...,...,...,...,...,...,...
495,496,22.350313,73.158712,5.397414,46.870101,70.737758,0.622544,1.169843e+06,13878.220979,1.094974,0.780683,1.673601
496,497,22.224220,73.325015,18.430132,5.000000,90.000000,0.300000,4.337719e+05,2000.000000,1.004889,0.706993,1.097356
497,498,22.291973,73.232467,5.936387,40.970436,67.307159,0.645154,1.170172e+06,14428.250427,1.072091,1.016573,0.663358
498,499,22.237151,73.135506,9.283541,31.952840,88.542423,0.496733,1.071397e+06,12143.418316,0.816669,0.505506,0.911794


In [9]:
# =========================
# 2. Clean Column Names (remove hidden spaces)
# =========================
temp_df.columns = temp_df.columns.str.strip()
urban_df.columns = urban_df.columns.str.strip()

# =========================
# 3. Automatically detect correct column names
# =========================
tree_cols = [col for col in urban_df.columns if "tree" in col.lower()]
asphalt_cols = [col for col in urban_df.columns if "asphalt" in col.lower()]

if len(tree_cols) == 0 or len(asphalt_cols) == 0:
    raise Exception("Tree or Asphalt column not found. Check column names.")

tree_col = tree_cols[0]
asphalt_col = asphalt_cols[0]

print("Using Tree Column:", tree_col)
print("Using Asphalt Column:", asphalt_col)
tree_col

Using Tree Column: tree_cover_pct
Using Asphalt Column: asphalt_pct


'tree_cover_pct'

In [13]:
# =========================
# 4. Merge Cleanly
# =========================
data = temp_df.merge(
    urban_df[["neighbourhood_id", tree_col, asphalt_col]],
    on="neighbourhood_id",
    how="inner"
)
data

,neighbourhood_id,date,latitude,longitude,distance_from_center_km,tree_cover_pct_x,asphalt_pct_x,building_density,median_income,population_density,...,avg_temp,max_temp,night_temp,surface_temp,humidity,wind_speed,solar_radiation,urban_heat_index,tree_cover_pct_y,asphalt_pct_y
0,1,2018-01-01,22.346937,73.255294,9.332579,32.950793,78.999617,0.356619,1.027265e+06,7087.025990,...,33.860953,38.670865,32.751726,43.340907,43.598772,11.782906,135.915229,3.917866,32.950793,78.999617
1,1,2018-01-02,22.346937,73.255294,9.332579,32.950793,78.999617,0.356619,1.027265e+06,7087.025990,...,34.024248,37.676170,32.519125,43.504202,65.758692,10.976173,721.332555,3.917866,32.950793,78.999617
2,1,2018-01-03,22.346937,73.255294,9.332579,32.950793,78.999617,0.356619,1.027265e+06,7087.025990,...,35.046252,38.932682,32.085145,44.526206,59.835979,10.338704,957.264957,3.917866,32.950793,78.999617
3,1,2018-01-04,22.346937,73.255294,9.332579,32.950793,78.999617,0.356619,1.027265e+06,7087.025990,...,36.132975,40.687233,34.237549,45.612929,61.777981,19.319173,688.575615,3.917866,32.950793,78.999617
4,1,2018-01-05,22.346937,73.255294,9.332579,32.950793,78.999617,0.356619,1.027265e+06,7087.025990,...,34.929092,39.794398,32.825062,44.409046,58.715489,4.521039,435.018377,3.917866,32.950793,78.999617
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,500,2023-06-19,22.196576,73.227007,13.290326,19.456538,90.000000,0.300000,4.599412e+05,7500.020972,...,37.489942,41.518964,34.482310,48.289942,72.948400,15.583221,149.067491,4.692594,19.456538,90.000000
999996,500,2023-06-20,22.196576,73.227007,13.290326,19.456538,90.000000,0.300000,4.599412e+05,7500.020972,...,36.563545,40.133282,34.274051,47.363545,70.154443,19.783045,290.324978,4.692594,19.456538,90.000000
999997,500,2023-06-21,22.196576,73.227007,13.290326,19.456538,90.000000,0.300000,4.599412e+05,7500.020972,...,35.957086,40.901301,33.841571,46.757086,53.190183,18.523549,182.109843,4.692594,19.456538,90.000000
999998,500,2023-06-22,22.196576,73.227007,13.290326,19.456538,90.000000,0.300000,4.599412e+05,7500.020972,...,37.214441,40.258155,35.591029,48.014441,71.979453,13.756614,810.852255,4.692594,19.456538,90.000000


In [18]:


# =========================
# 5. Define Regression Variables
# =========================
y = data["surface_temp"]
X = data[["tree_cover_pct_x", "asphalt_pct_x"]]
X = sm.add_constant(X)

# =========================
# 6. Quantile Regression
# =========================
quantiles = [0.50, 0.75, 0.90]

for q in quantiles:
    model = sm.QuantReg(y, X).fit(q=q)

    print("\n==============================")
    print(f"Quantile: {q}")
    print("==============================")
    print(model.params)


Quantile: 0.5
const               35.952032
tree_cover_pct_x    -0.045357
asphalt_pct_x        0.140947
dtype: float64

Quantile: 0.75
const               42.089496
tree_cover_pct_x    -0.045874
asphalt_pct_x        0.139980
dtype: float64

Quantile: 0.9
const               44.288259
tree_cover_pct_x    -0.046332
asphalt_pct_x        0.141055
dtype: float64
